# TWPA Workflow

Workflow:
1. Scan TWPA flux with pump off / reference.
2. Fix pump power, sweep pump frequency and Yoko flux.
3. Fix flux, sweep pump power and pump frequency.

In [ ]:
import numpy as np

from qick.asm_v2 import QickSweep1D

from QickworkspaceV2.config.system_cfg import DATA_PATH, ExperimentConfig, config_list
from QickworkspaceV2.core.base_experiment import BaseExperiment
from QickworkspaceV2.experiments.twpa import TWPAFlux, TWPAGain, TWPAGainPower
from QickworkspaceV2.instruments import BaseInstrumentManager
from QickworkspaceV2.instruments.mg3692 import AnritsuMG3692

## Optional QICK Session

Skip this cell if `BaseExperiment.setup(...)` or `connect_pyro4(...)` is already active in this kernel.

In [ ]:
# Uncomment and edit if this notebook starts from a fresh kernel.
# soc, soccfg = BaseExperiment.connect_pyro4(
#     ns_host="192.168.10.82",
#     ns_port=8888,
#     proxy_name="myqick",
#     data_path=DATA_PATH,
# )

## Instruments

In [ ]:
yoko_connect = "USB0::0x0B21::0x0039::91S522309::INSTR"
pump = AnritsuMG3692("192.168.10.182")

YOKO_NAME = "twpa_flux"
YOKO_MODE = "current"  # "current" or "voltage"

inst = BaseInstrumentManager()
yoko = inst.add_yoko(
    YOKO_NAME,
    yoko_connect,
    auto_limits=False,
    limits={"current": (-3e-3, 3e-3), "voltage": (-2.0, 2.0)},
    current_ramp_step=1e-8,
    voltage_ramp_step=1e-5,
    ramp_interval=0.01,
)

print(inst.status)

## Shared TWPA Config

Set the qubit/config index and the readout frequency span used for all TWPA scans.

In [ ]:
qubit = "Q1"  # Change if needed, e.g. "Q3" or 2
config_all = ExperimentConfig(config_list)
run_cfg = config_all.get_qubit(qubit)

PY_AVG = 5

# QICK readout frequency sweep in MHz.
CENTER_FREQ_MHZ = run_cfg["res_freq_ge"]
SPAN_MHZ = 400
FREQ_STEPS = 401
FREQ_START_MHZ = CENTER_FREQ_MHZ - SPAN_MHZ / 2
FREQ_STOP_MHZ = CENTER_FREQ_MHZ + SPAN_MHZ / 2

# Yoko flux sweep. Units follow YOKO_MODE above.
YOKO_START = -0.5e-3
YOKO_STOP = 0.5e-3
YOKO_STEPS = 101
YOKO_VALUES = np.linspace(YOKO_START, YOKO_STOP, YOKO_STEPS)

run_cfg.update(
    [
        ("steps", FREQ_STEPS),
        ("res_freq_ge", QickSweep1D("freqloop", FREQ_START_MHZ, FREQ_STOP_MHZ)),
        ("yoko_value", YOKO_VALUES),
    ]
)

print(f"Frequency sweep: {FREQ_START_MHZ:.3f} to {FREQ_STOP_MHZ:.3f} MHz")
print(f"Yoko sweep: {YOKO_START:.6g} to {YOKO_STOP:.6g} ({YOKO_MODE})")

## 1. TWPA Flux Scan

This creates the pump-off/reference map. Keep `twpa_flux_ref` for later gain normalization.

In [ ]:
pump.off()

twpa_flux_ref = TWPAFlux(run_cfg)
twpa_flux_ref.run(
    PY_AVG,
    instrument_manager=inst,
    yoko_name=YOKO_NAME,
    yoko_value=YOKO_VALUES,
    yoko_mode=YOKO_MODE,
)

twpa_flux_ref.plot(normalize=False)
twpa_flux_ref.saveNetCDF(filename="twpa_flux_reference")
twpa_flux_ref.saveLabber(qubit, config_all=config_all, title="reference")

## 2. Fixed Pump Power: Sweep Pump Frequency and Yoko Flux

In [ ]:
PUMP_POWER_DBM = -10.0
PUMP_FREQ_START_HZ = 6.0e9
PUMP_FREQ_STOP_HZ = 8.0e9
PUMP_FREQ_STEPS = 81
PUMP_FREQS = np.linspace(PUMP_FREQ_START_HZ, PUMP_FREQ_STOP_HZ, PUMP_FREQ_STEPS)

twpa_gain = TWPAGain(
    run_cfg,
    pump_source=pump,
    pump_freqs=PUMP_FREQS,
    pump_power=PUMP_POWER_DBM,
)

twpa_gain.run(
    PY_AVG,
    instrument_manager=inst,
    yoko_name=YOKO_NAME,
    yoko_value=YOKO_VALUES,
    yoko_mode=YOKO_MODE,
    temp_folder=DATA_PATH,
    reference=twpa_flux_ref,
)

gain_path, gain_ref_path = twpa_gain.saveNetCDF(
    reference=twpa_flux_ref,
    filename="twpa_gain_fixed_power",
)
print(gain_path, gain_ref_path)
twpa_gain.saveLabber(qubit, config_all=config_all, title="fixed_power")

In [ ]:
best_points, total_score = twpa_gain.analyze(
    reference=twpa_flux_ref,
    gain_min=12,
    gain_median=15,
    ripple_max=5,
    f_min=4e9,
    f_max=8e9,
    n_best=5,
)

best_points

## 3. Fixed Flux: Sweep Pump Power and Pump Frequency

Use the best flux from the previous analysis, or manually set `FIXED_FLUX`.

In [ ]:
if "best_points" in globals() and len(best_points) > 0:
    FIXED_FLUX = float(best_points[0]["ifbl"])
else:
    FIXED_FLUX = 0.0

# Override manually if desired:
# FIXED_FLUX = 120e-6

PUMP_POWER_START_DBM = -20.0
PUMP_POWER_STOP_DBM = 0.0
PUMP_POWER_STEPS = 41
PUMP_POWERS = np.linspace(PUMP_POWER_START_DBM, PUMP_POWER_STOP_DBM, PUMP_POWER_STEPS)

print(f"Fixed flux = {FIXED_FLUX:.6g} ({YOKO_MODE})")

In [ ]:
twpa_gain_power = TWPAGainPower(
    run_cfg,
    pump_source=pump,
    pump_freqs=PUMP_FREQS,
    pump_powers=PUMP_POWERS,
)

twpa_gain_power.run(
    PY_AVG,
    instrument_manager=inst,
    yoko_name=YOKO_NAME,
    yoko_value=np.array([FIXED_FLUX]),
    yoko_mode=YOKO_MODE,
    temp_folder=DATA_PATH,
    reference=twpa_flux_ref,
)

gain_power_path, gain_power_ref_path = twpa_gain_power.saveNetCDF(
    reference=twpa_flux_ref,
    filename="twpa_gain_power_fixed_flux",
)
print(gain_power_path, gain_power_ref_path)
twpa_gain_power.saveLabber(qubit, config_all=config_all, title="fixed_flux")

In [ ]:
power_results = twpa_gain_power.analyze(
    reference=twpa_flux_ref,
    gain_min=12,
    gain_median=15,
    ripple_max=5,
    f_min=4e9,
    f_max=8e9,
)

power_results

## Cleanup

In [ ]:
pump.off()
# inst.off(YOKO_NAME)  # Uncomment if desired.